In [1]:
import pandas as pd
import re
import json


input_path = "jaemin1_.txt"  

with open(input_path, "r", encoding="utf-8") as f:
    text = f.read()

pattern = r"(\/[^\n]+\/(normal|abnormal)\/[^\n]+)\s*[\s\S]*?(?:```json\s*([\s\S]*?)\s*```|(\{[\s\S]*?\}))"
matches = re.findall(pattern, text, re.DOTALL)

rows = []

for match in matches:
    
    path, truth, json_block, json_raw = match
    json_str = json_block if json_block else json_raw
    
    try:
        data = json.loads(json_str)

        is_scam = bool(data.get("is_scam"))

        if (is_scam and truth == "abnormal") or (not is_scam and truth == "normal"):
            result = True
        else:
            result = False

        rows.append({
            "filename": path.split("/")[-1].strip(),
            "truth": truth,
            "is_scam": data.get("is_scam"),
            "result": result,
            "is_correct": data.get("is_scam"),
            "confidence": data.get("confidence"),
            "risk": data.get("risk"),
            "evidence": "; ".join(data.get("evidence", [])),
            "explanation": data.get("explanation"),            
        })
    except json.JSONDecodeError as e:
        print(f"JSON 파싱 오류 발생 ({path}): {e}")

df = pd.DataFrame(rows, columns=["filename", "truth", "is_scam", "result", "confidence", "risk", "evidence", "explanation"])

output_path = "scam_results.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"CSV 파일 생성 완료: {output_path}")
print(df)


CSV 파일 생성 완료: scam_results.csv
    filename     truth  is_scam  result  confidence  risk  \
0   0025.mp4    normal     True   False        0.90  high   
1   0011.mp4    normal    False    True        0.65   low   
2   0024.mp4    normal    False    True        0.50   low   
3   0004.mp4    normal    False    True        0.35   low   
4   0001.mp4    normal    False    True        0.25   low   
5   0006.mp4    normal    False    True        0.52   low   
6   0016.mp4    normal    False    True        0.20   low   
7   0005.mp4    normal     True   False        0.92  high   
8   0018.mp4    normal    False    True        0.65   low   
9   0023.mp4    normal    False    True        0.43   low   
10  0007.mp4    normal    False    True        0.35   low   
11  0014.mp4    normal    False    True        0.65   low   
12  0021.mp4    normal    False    True        0.65   low   
13  0027.mp4    normal    False    True        0.25   low   
14  0012.mp4    normal    False    True        0.35   

In [5]:
df.groupby(["truth", "result"]).agg({"filename":"count"})

filename
truth    result          
abnormal False          6
         True          21
normal   False          4
         True          24

In [6]:
df[df['result']==True]

,filename,truth,is_scam,result,confidence,risk,evidence,explanation
1,0011.mp4,normal,False,True,0.65,low,Payment requests; External contact request,The video shows typical actions one might do d...
2,0024.mp4,normal,False,True,0.50,low,이런 곳도 있을네..?,The video demonstrates the convenience of a pr...
3,0004.mp4,normal,False,True,0.35,low,아오아오거거거; 에이에이에이에이에이,The video contains humorous and light-hearted ...
4,0001.mp4,normal,False,True,0.25,low,Minimum wage increases; Student internships,The content appears to be discussing labor con...
5,0006.mp4,normal,False,True,0.52,low,successful surgery; pleas for understanding,The description indicates a personal experienc...
6,0016.mp4,normal,False,True,0.20,low,고작한 장난을 한번 쳐볼게요; 다시 쓰고 싶어..,The video describes a Rubik's cube solving cha...
8,0018.mp4,normal,False,True,0.65,low,한번ONLY,The video appears to be a casual review of var...
9,0023.mp4,normal,False,True,0.43,low,nonstick coating; Noodle stir-fry,The video description appears to be promoting ...
10,0007.mp4,normal,False,True,0.35,low,"住了几天, 每天工作7小时; 每天赚了多少钱",The video details a personal experience of wor...
11,0014.mp4,normal,False,True,0.65,low,Ice is added and the drink is poured in.; The ...,The video appears to be demonstrating how to m...


In [7]:
df[df['result']==False]

,filename,truth,is_scam,result,confidence,risk,evidence,explanation
0,0025.mp4,normal,True,False,0.90,high,garantir um profit de 50% no taxi Uber; convi...,O vídeo apresenta promessas de lucro garantido...
7,0005.mp4,normal,True,False,0.92,high,false sense of urgency; pressure to act quickl...,The video contains phrases like 'Don't miss ou...
18,0002.mp4,normal,True,False,0.90,high,MBTI별 추천알바; MBTI별 추천알바,The video mentions job recommendations based o...
22,0026.mp4,normal,True,False,0.95,high,우배 택시 밸 PIT; 가능한 부가가치,The video contains text suggesting a scam with...
28,0025.mp4,abnormal,False,False,0.35,low,,The video description does not contain any exp...
33,0004.mp4,abnormal,False,False,0.45,low,Various sandals displayed; Product demonstration,The video appears to be a normal product showc...
38,0005.mp4,abnormal,False,False,0.32,low,gaesanghwape gugmaedaeheung mojjeon; handaimeu...,The video shows a nighttime street scene with ...
39,0018.mp4,abnormal,False,False,0.35,low,,The content appears to be a social media conve...
43,0012.mp4,abnormal,False,False,0.65,low,normal presentation of jeans; clear product in...,The video shows a normal demonstration and des...
51,0000.mp4,abnormal,False,False,0.35,low,고객님께서 주문하신 남성 패딩 배송전实事已然; 여ørn 패딩 필요한분들 문의해주신,The description is promoting a product and dis...
